In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pickle
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import GridSearchCV
from xgboost import XGBRegressor
from xgboost.callback import EarlyStopping
from sklearn.metrics import root_mean_squared_error, r2_score, mean_squared_error, mean_absolute_percentage_error

In [2]:
X_train = pd.read_csv('../data/processed/X_train.csv')
X_test = pd.read_csv('../data/processed/X_test.csv')
y_train = pd.read_csv('../data/processed/y_train.csv')
y_test = pd.read_csv('../data/processed/y_test.csv')

In [3]:
forest = RandomForestRegressor(random_state=42)
forest.fit(X_train, y_train)

c:\Users\J Vicente\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\base.py:1365: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs)


,n_estimators,100
,criterion,'squared_error'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,1.0
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [ ]:
pred_forest = forest.predict(X_test)
pred_forest

In [ ]:
y_train

,total_cost
0,81.53
1,524.24
2,65.17
3,59.52
4,101.95
...,...
1599995,1189.80
1599996,184.14
1599997,142.78
1599998,1231.56


In [ ]:
early = EarlyStopping(
    rounds=3,
    save_best=True,
    maximize=False,
    metric_name="root_mean_squared_error",
)

boost = XGBRegressor(tree_method="hist", eval_metric=root_mean_squared_error,random_state=42)
boost.fit(X_train, y_train)

,objective,'reg:squarederror'
,base_score,None
,booster,None
,callbacks,None
,colsample_bylevel,None
,colsample_bynode,None
,colsample_bytree,None
,device,None
,early_stopping_rounds,None
,enable_categorical,False
,eval_metric,<function roo...002CF6E094CC0>


In [ ]:
pred_boost = boost.predict(X_test)

In [ ]:
y_test['pred_boost'] = pred_boost
y_test['pred_forest'] = pred_forest
y_test

,total_cost,pred_boost,pred_forest
0,537.01,1215.766724,1256.2964
1,1217.11,1207.236206,1212.7227
2,246.05,197.577805,203.2439
3,1422.59,1219.485596,1197.2071
4,678.26,603.513916,615.6990
...,...,...,...
399995,118.99,150.166061,157.3350
399996,127.73,149.230652,162.3077
399997,195.35,197.272263,210.4844
399998,128.74,151.128906,151.7352


In [ ]:
metrics_optimized = {'MSE': (mean_squared_error(y_test['total_cost'], pred_forest), mean_squared_error(y_test['total_cost'], pred_boost)),
           'RMSE': (root_mean_squared_error(y_test['total_cost'], pred_forest), root_mean_squared_error(y_test['total_cost'], pred_boost)),
           'MAPE' : (mean_absolute_percentage_error(y_test['total_cost'], pred_forest), mean_absolute_percentage_error(y_test['total_cost'], pred_boost)),
           'R^2': (r2_score(y_test['total_cost'], pred_forest), r2_score(y_test['total_cost'], pred_boost)),}

metrics_optimized

{'MSE': (25993.03903147748, 25414.85698443732),
 'RMSE': (161.22356847395943, 159.42037819688335),
 'MAPE': (0.32303370786722047, 0.3205117597486905),
 'R^2': (0.8434836659474729, 0.8469651724503533)}

In [ ]:
with open('../models/forest.pkl', 'wb') as file:
    pickle.dump(forest, file)

with open('../models/boost.pkl', 'wb') as file:
    pickle.dump(boost, file)

In [ ]:
params = {
    'n_estimators': [100, 250, 500],
    'max_depth': [3,  5, 7],
    'learning_rate': [0.01, 0.05, 0.1],
    'subsample': [0.8, 0.9, 1.0],
}

In [ ]:
grid = GridSearchCV(boost, params)
grid.fit(X_train, y_train)

,estimator,"XGBRegressor(...ree=None, ...)"
,param_grid,"{'learning_rate': [0.01, 0.05, ...], 'max_depth': [3, 5, ...], 'n_estimators': [100, 250, ...], 'subsample': [0.8, 0.9, ...]}"
,scoring,None
,n_jobs,None
,refit,True
,cv,None
,verbose,0
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,objective,'reg:squarederror'


In [ ]:
pred_grid_boost = grid.predict(X_test)
pred_grid_boost

array([1191.7274 , 1201.6626 ,  200.8801 , ...,  200.31686,  150.6846 ,
        599.2941 ], shape=(400000,), dtype=float32)

In [ ]:
metrics_optimized = {'MSE': (mean_squared_error(y_test['total_cost'], pred_boost), mean_squared_error(y_test['total_cost'], pred_grid_boost)),
           'RMSE': (root_mean_squared_error(y_test['total_cost'], pred_boost), root_mean_squared_error(y_test['total_cost'], pred_grid_boost)),
           'MAPE' : (mean_absolute_percentage_error(y_test['total_cost'], pred_boost), mean_absolute_percentage_error(y_test['total_cost'], pred_grid_boost)),
           'R^2': (r2_score(y_test['total_cost'], pred_boost), r2_score(y_test['total_cost'], pred_grid_boost)),}

metrics_optimized

{'MSE': (25414.85698443732, 25316.173516091174),
 'RMSE': (159.42037819688335, 159.11057009542506),
 'MAPE': (0.3205117597486905, 0.32027447205459075),
 'R^2': (0.8469651724503533, 0.8475593921057935)}

In [ ]:
y_test['pred_boost'] = pred_boost
y_test['pred_grid_boost'] = pred_grid_boost
y_test

,total_cost,pred_boost,pred_forest,pred_grid_boost
0,537.01,1215.766724,1256.2964,1191.727417
1,1217.11,1207.236206,1212.7227,1201.662598
2,246.05,197.577805,203.2439,200.880096
3,1422.59,1219.485596,1197.2071,1204.920166
4,678.26,603.513916,615.6990,598.098267
...,...,...,...,...
399995,118.99,150.166061,157.3350,149.597595
399996,127.73,149.230652,162.3077,149.622131
399997,195.35,197.272263,210.4844,200.316864
399998,128.74,151.128906,151.7352,150.684601


In [ ]:
with open('../models/grid_boost.pkl', 'wb') as file:
    pickle.dump(grid, file)